# 16 - Polars operaciones — **Soluciones**

In [ ]:

import numpy as np, pandas as pd
try:
    import polars as pl; HAS_POLARS=True
except: HAS_POLARS=False
rng=np.random.default_rng(7); n=2000
ventas=pd.DataFrame({
    "id": np.arange(n),
    "producto": rng.choice(["A","B","C"], n),
    "precio": rng.normal(30,8,n).round(2),
    "unidades": rng.integers(1,6,n),
    "tienda": rng.choice(["Norte","Sur","Este","Oeste"], n)
})
productos=pd.DataFrame({"producto":["A","B","C"],"categoria":["Hogar","Electrónica","Ocio"]})
if HAS_POLARS:
    v=pl.from_pandas(ventas); p=pl.from_pandas(productos)
    joined=v.join(p,on="producto",how="left").with_columns((pl.col("precio")*pl.col("unidades")).alias("importe"))
    agg=(joined.groupby(["categoria","tienda"])
         .agg([pl.sum("importe").alias("ventas_totales"),
               pl.mean("precio").alias("precio_medio"),
               pl.count().alias("n")]))
    tabla=agg.pivot(values="ventas_totales", index="categoria", columns="tienda").fill_null(0)
else:
    joined=ventas.merge(productos,on="producto",how="left")
    joined["importe"]=joined["precio"]*joined["unidades"]
    agg=(joined.groupby(["categoria","tienda"])
         .agg(ventas_totales=("importe","sum"), precio_medio=("precio","mean"), n=("id","count"))
         .reset_index())
    tabla=agg.pivot(index="categoria", columns="tienda", values="ventas_totales").fillna(0)



---

## Tests automáticos


In [ ]:

import numpy as np, pandas as pd
try:
    import polars as pl; HAS_POLARS=True
except: HAS_POLARS=False
rng=np.random.default_rng(7); n=2000
ventas=pd.DataFrame({
    "id": np.arange(n),
    "producto": rng.choice(["A","B","C"], n),
    "precio": rng.normal(30,8,n).round(2),
    "unidades": rng.integers(1,6,n),
    "tienda": rng.choice(["Norte","Sur","Este","Oeste"], n)
})
productos=pd.DataFrame({"producto":["A","B","C"],"categoria":["Hogar","Electrónica","Ocio"]})
if HAS_POLARS:
    v=pl.from_pandas(ventas); p=pl.from_pandas(productos)
    joined=v.join(p,on="producto",how="left").with_columns((pl.col("precio")*pl.col("unidades")).alias("importe"))
    agg=(joined.groupby(["categoria","tienda"])
         .agg([pl.sum("importe").alias("ventas_totales"), pl.mean("precio").alias("precio_medio"), pl.count().alias("n")]))
    tabla=agg.pivot(values="ventas_totales", index="categoria", columns="tienda").fill_null(0)
else:
    joined=ventas.merge(productos,on="producto",how="left")
    joined["importe"]=joined["precio"]*joined["unidades"]
    agg=(joined.groupby(["categoria","tienda"])
         .agg(ventas_totales=("importe","sum"), precio_medio=("precio","mean"), n=("id","count"))
         .reset_index())
    tabla=agg.pivot(index="categoria", columns="tienda", values="ventas_totales").fillna(0)


In [ ]:

def check(c,o,f):
    if c: print("✔",o)
    else: raise AssertionError("✘ "+f)

# tabla a pandas si es polars
if hasattr(tabla, "to_pandas"):
    t = tabla.to_pandas()
else:
    t = tabla

check(t.shape[0]>0 and t.shape[1]>0, "Tabla pivot creada.", "Tabla pivot vacía.")
check(not t.isna().any().any(), "Tabla sin NA.", "Hay NA en la tabla pivot.")
